# MNIST MLP3 — Muon + auxiliary AdamW baseline

This is the reference Muon control. `fc1.weight` and `fc2.weight` use Muon with step-level warm-up/cosine decay from 0.02 to 0.002, momentum 0.95, Nesterov, five Newton–Schulz iterations, and weight decay 0.01. `fc3.weight` and all biases use auxiliary AdamW from 3e-4 to 3e-5 with betas (0.90, 0.95). The historical file and directory key remains `sgd_momentum_muon` only for compatibility.

Results are written beneath the versioned suite directory `mnist_mlp3_recipe_v3`. Legacy unversioned result directories—including the unstable 20-epoch Muon + auxiliary-SGD run—are ignored and cannot be resumed or compared accidentally.

The data policy is 55,000 optimization examples, a fixed 5,000-example validation split, and the official test set used only for monitoring. At epoch zero and every epoch the code runs `watcher.analyze(ERG=True, randomize=True)` and retains direct `alpha`, `ERG_gap`, and `num_traps`.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a current clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    MNIST_REFERENCE_RECIPE_VERSION,
    MNIST_REFERENCE_SUITE_SLUG,
    plot_all_replicates,
    run_baseline_replicates,
)

BASE_RUN_ROOT = Path(
    os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')
).expanduser().resolve()
RUN_ROOT = BASE_RUN_ROOT / MNIST_REFERENCE_SUITE_SLUG
DATA_DIR = Path(
    os.environ.get('RG_BASELINE_DATA_DIR', ROOT / 'data')
).expanduser().resolve()
for directory in (BASE_RUN_ROOT, RUN_ROOT, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
CONFIG = BaselineConfig(
    optimizer='sgd_momentum_muon',
    epochs=30,
    validation_size=5_000,
    muon_parameter_names=('fc1.weight', 'fc2.weight'),
    muon_learning_rate=0.02,
    muon_min_learning_rate=0.002,
    muon_warmup_epochs=2,
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_weight_decay=0.01,
    muon_newton_schulz_steps=5,
    muon_aux_learning_rate=3e-4,
    muon_aux_min_learning_rate=3e-5,
    muon_aux_beta1=0.90,
    muon_aux_beta2=0.95,
    muon_aux_weight_decay=0.01,
    ww_randomize=True,
    save_epoch_checkpoints=True,
)
CONFIG.validate()
if CONFIG.recipe_version != MNIST_REFERENCE_RECIPE_VERSION:
    raise RuntimeError(
        f'Notebook expects MNIST recipe v{MNIST_REFERENCE_RECIPE_VERSION}; '
        f'loaded v{CONFIG.recipe_version}. Pull current main before running.'
    )
assert CONFIG.muon_aux_learning_rate == 3e-4
assert CONFIG.muon_weight_decay == 0.01

SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
RUN_DIR = RUN_ROOT / CONFIG.run_slug
PLOT_DIR = RUN_DIR / 'plots'
for directory in (RUN_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LEGACY_DIR = BASE_RUN_ROOT / CONFIG.run_slug
if LEGACY_DIR.is_dir() and LEGACY_DIR != RUN_DIR:
    print('Ignoring legacy unversioned artifacts:', LEGACY_DIR)
print('MNIST reference recipe:', CONFIG.recipe_version)
print('optimizer implementation:', CONFIG.optimizer_label)
print('versioned suite root:', RUN_ROOT)
print('current run directory:', RUN_DIR)
display(pd.DataFrame([CONFIG.__dict__]))


## Run or resume three complete trajectories

The runner asserts and records the Muon/AdamW parameter assignment. Only checkpoints carrying the current recipe fingerprint inside the versioned suite directory can resume.


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
    resume=True,
    overwrite=False,
)
assert suite.config_template.recipe_version == MNIST_REFERENCE_RECIPE_VERSION
assert set(suite.optimizer_groups['kind']) == {
    'muon', 'adamw_decay', 'adamw_no_decay'
}
plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)
display(
    suite.optimizer_groups.sort_values(
        ['seed', 'epoch', 'group_index']
    ).head(12)
)


## Train, validation, test, and learning-rate trajectories

The primary Muon and auxiliary AdamW schedules are logged separately. Validation loss is the only checkpoint-selection signal.


In [ ]:
for metric in [
    'train_loss', 'validation_loss', 'test_loss',
    'train_accuracy', 'validation_accuracy', 'test_accuracy',
    'primary_lr', 'auxiliary_lr',
]:
    summary = suite.performance_summary[
        suite.performance_summary['metric'].eq(metric)
    ].sort_values('epoch')
    if summary.empty:
        continue
    figure, axis = plt.subplots(figsize=(9, 5))
    for seed, run in suite.performance.groupby('seed'):
        axis.plot(run['epoch'], run[metric], alpha=0.18, linewidth=0.8)
    axis.plot(summary['epoch'], summary['mean'], linewidth=2.0, label='mean')
    axis.fill_between(
        summary['epoch'], summary['ci_low'], summary['ci_high'], alpha=0.16
    )
    axis.set(
        xlabel='Epoch',
        ylabel=metric.replace('_', ' ').title(),
        title=f'{CONFIG.optimizer_label}: {metric}',
    )
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(
        PLOT_DIR / f'{metric}_95ci.png', dpi=170, bbox_inches='tight'
    )
    plt.show()

display(
    suite.performance_summary[
        suite.performance_summary['metric'].isin([
            'train_loss', 'validation_loss', 'test_loss',
            'train_accuracy', 'validation_accuracy', 'test_accuracy',
            'primary_lr', 'auxiliary_lr',
        ])
    ].sort_values(['metric', 'epoch'])
)


## Layerwise WeightWatcher diagnostics


In [ ]:
required = [
    'alpha', 'ERG_gap', 'num_traps',
    'm_midpoint', 'trace_log_midpoint_per_eval',
]
rows = suite.spectral_summary[
    suite.spectral_summary['metric'].isin(required)
].sort_values(['metric', 'layer', 'epoch'])
assert rows['n'].eq(3).all()
display(rows)


In [ ]:
required_paths = [
    RUN_DIR / 'performance_by_epoch_and_seed.csv',
    RUN_DIR / 'spectral_metrics_by_epoch_layer_and_seed.csv',
    RUN_DIR / 'performance_summary_95ci.csv',
    RUN_DIR / 'spectral_summary_95ci.csv',
    RUN_DIR / 'replicate_manifest.json',
    PLOT_DIR / '7_layerwise_weightwatcher_num_traps_95ci.png',
]
for seed in SEEDS:
    seed_dir = RUN_DIR / 'seeds' / f'seed_{seed}'
    required_paths.extend([
        seed_dir / 'checkpoint_latest.pt',
        seed_dir / 'checkpoint_best.pt',
        seed_dir / 'final_state.pt',
        seed_dir / 'test_results.json',
        seed_dir / 'run_complete.json',
    ])
missing = [path for path in required_paths if not path.is_file()]
if LEGACY_DIR.is_dir() and LEGACY_DIR != RUN_DIR:
    print('Ignoring legacy unversioned artifacts:', LEGACY_DIR)
print('MNIST reference recipe:', CONFIG.recipe_version)
print('optimizer implementation:', CONFIG.optimizer_label)
print('versioned suite root:', RUN_ROOT)
print('current run directory:', RUN_DIR)
display(pd.DataFrame([CONFIG.__dict__]))


## Run or resume three complete trajectories

The runner asserts and records the Muon/AdamW parameter assignment. Only checkpoints carrying the current recipe fingerprint inside the versioned suite directory can resume.


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
    resume=True,
    overwrite=False,
)
assert suite.config_template.recipe_version == MNIST_REFERENCE_RECIPE_VERSION
assert set(suite.optimizer_groups['kind']) == {
    'muon', 'adamw_decay', 'adamw_no_decay'
}
plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)
display(
    suite.optimizer_groups.sort_values(
        ['seed', 'epoch', 'group_index']
    ).head(12)
)


## Train, validation, test, and learning-rate trajectories

The primary Muon and auxiliary AdamW schedules are logged separately. Validation loss is the only checkpoint-selection signal.


In [ ]:
for metric in [
    'train_loss', 'validation_loss', 'test_loss',
    'train_accuracy', 'validation_accuracy', 'test_accuracy',
    'primary_lr', 'auxiliary_lr',
]:
    summary = suite.performance_summary[
        suite.performance_summary['metric'].eq(metric)
    ].sort_values('epoch')
    if summary.empty:
        continue
    figure, axis = plt.subplots(figsize=(9, 5))
    for seed, run in suite.performance.groupby('seed'):
        axis.plot(run['epoch'], run[metric], alpha=0.18, linewidth=0.8)
    axis.plot(summary['epoch'], summary['mean'], linewidth=2.0, label='mean')
    axis.fill_between(
        summary['epoch'], summary['ci_low'], summary['ci_high'], alpha=0.16
    )
    axis.set(
        xlabel='Epoch',
        ylabel=metric.replace('_', ' ').title(),
        title=f'{CONFIG.optimizer_label}: {metric}',
    )
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(
        PLOT_DIR / f'{metric}_95ci.png', dpi=170, bbox_inches='tight'
    )
    plt.show()

display(
    suite.performance_summary[
        suite.performance_summary['metric'].isin([
            'train_loss', 'validation_loss', 'test_loss',
            'train_accuracy', 'validation_accuracy', 'test_accuracy',
            'primary_lr', 'auxiliary_lr',
        ])
    ].sort_values(['metric', 'epoch'])
)


## Layerwise WeightWatcher diagnostics


In [ ]:
required = [
    'alpha', 'ERG_gap', 'num_traps',
    'm_midpoint', 'trace_log_midpoint_per_eval',
]
rows = suite.spectral_summary[
    suite.spectral_summary['metric'].isin(required)
].sort_values(['metric', 'layer', 'epoch'])
assert rows['n'].eq(3).all()
display(rows)


In [ ]:
required_paths = [
    RUN_DIR / 'performance_by_epoch_and_seed.csv',
    RUN_DIR / 'spectral_metrics_by_epoch_layer_and_seed.csv',
    RUN_DIR / 'performance_summary_95ci.csv',
    RUN_DIR / 'spectral_summary_95ci.csv',
    RUN_DIR / 'replicate_manifest.json',
    PLOT_DIR / '7_layerwise_weightwatcher_num_traps_95ci.png',
]
for seed in SEEDS:
    seed_dir = RUN_DIR / 'seeds' / f'seed_{seed}'
    required_paths.extend([
        seed_dir / 'checkpoint_latest.pt',
        seed_dir / 'checkpoint_best.pt',
        seed_dir / 'final_state.pt',
        seed_dir / 'test_results.json',
        seed_dir / 'run_complete.json',
    ])
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise RuntimeError(
        'Missing required artifacts:\n' + '\n'.join(map(str, missing))
    )
print('verified artifacts:', len(required_paths))
